In [1]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../data/sepsis.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:group": "string",
        "case:age": "float32",
        "Leucocytes": "float32",
        "CRP": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,CRP,Leucocytes,case:age,concept:name,lifecycle:transition,org:group,time_delta
0,A,2014-10-22 11:15:41,0.0,0.0,85.0,ER Registration,complete,A,0
1,A,2014-10-22 11:27:00,0.0,9.6,85.0,Leucocytes,complete,B,679
2,A,2014-10-22 11:27:00,21.0,0.0,85.0,CRP,complete,B,0
3,A,2014-10-22 11:27:00,0.0,0.0,85.0,LacticAcid,complete,B,0
4,A,2014-10-22 11:33:37,0.0,0.0,85.0,ER Triage,complete,C,397
5,A,2014-10-22 11:34:00,0.0,0.0,85.0,ER Sepsis Triage,complete,A,23
6,A,2014-10-22 14:03:47,0.0,0.0,85.0,IV Liquid,complete,A,8987
7,A,2014-10-22 14:03:47,0.0,0.0,85.0,IV Antibiotics,complete,A,0
8,A,2014-10-22 14:13:19,0.0,0.0,85.0,Admission NC,complete,D,572
9,A,2014-10-24 09:00:00,109.0,0.0,85.0,CRP,complete,B,154001


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['CRP', 'Leucocytes', 'case:age', 'concept:name', 'lifecycle:transition', 'org:group', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
org:group                      categorical    event    yes    ['A', 'B', 'C', ...]                     N/A        data_derived        
case:age                       continuous     case     yes    [40.00, 90.00]                           10.0000    quantile_derived    
time_delta                     continuous     event    yes    [0.00, 38748.80]                         139.0000   quantile_derived    
Leucocytes                    

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[{'Admission NC',
  'CRP',
  'ER Registration',
  'ER Sepsis Triage',
  'ER Triage',
  'IV Antibiotics',
  'IV Liquid',
  'LacticAcid',
  'Leucocytes'},
 {'ER Registration', 'ER Sepsis Triage', 'ER Triage', 'IV Antibiotics'}]

In [13]:
engine.branching_sets

[{'Admission NC',
  'CRP',
  'ER Registration',
  'ER Sepsis Triage',
  'ER Triage',
  'IV Antibiotics',
  'IV Liquid',
  'LacticAcid',
  'Leucocytes'},
 {'ER Registration', 'ER Sepsis Triage', 'ER Triage', 'IV Antibiotics'},
 {'Release C', 'Release D', 'Release E'}]

### --- Experiments Generation ---

In [14]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/sepsis-cf_seed777_experiments_ga_output.txt", console=False)

In [15]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [16]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

In [17]:
exp_df_mul, metadata_mul = ExperimentHandler.load("../experiments/cf_generated_experiments_multiple_desired")
print("Mined using parameters:", metadata_mul["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/190 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,FFA,3,1,3,0.583186,0.341371,0.825000,0.577778,0.000000,...,0.361111,0.000000,0.250000,0.500000,0.000000,0.111111,0.0,0.000000,0.0,0.000000
1,0,CL,4,1,4,0.436126,0.147252,0.725000,0.411111,0.181818,...,0.774144,0.181818,0.258992,0.500000,0.017985,0.333333,0.0,0.866484,0.0,1.000000
2,0,KY,5,1,5,0.466237,0.107474,0.825000,0.366667,0.303846,...,0.787767,0.307692,0.257852,0.500000,0.015705,0.222222,0.0,0.884112,0.0,1.000000
3,0,FP,6,1,6,0.528638,0.107276,0.950000,0.361111,0.400000,...,0.761111,0.400000,0.250000,0.500000,0.000000,0.111111,0.0,0.887878,0.0,1.000000
4,0,GIA,7,1,7,0.490287,0.205575,0.775000,0.444444,0.235294,...,0.773351,0.117647,0.322370,0.500000,0.144741,0.333333,0.0,0.882502,0.0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,18,MFA,12,1,11,0.368097,0.177102,0.559091,0.430000,0.370370,...,0.209682,0.185185,0.002275,0.000000,0.004550,0.022222,0.0,0.905541,0.0,1.000000
163,18,OKA,12,1,11,0.295402,0.145349,0.445455,0.520000,0.298148,...,0.527203,0.000000,0.149425,0.181818,0.117033,0.377778,0.0,0.768012,0.0,0.999999
164,18,SS,12,1,11,0.342328,0.189202,0.495455,0.511111,0.296296,...,0.468715,0.185185,0.083529,0.090909,0.076150,0.200000,0.0,0.889460,0.0,1.000000
165,18,ZFA,12,1,11,0.340859,0.168082,0.513636,0.457778,0.327778,...,0.103592,0.000000,0.059147,0.090909,0.027385,0.044444,0.0,0.000000,0.0,0.000000


In [21]:
results_mul = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_multiple_desired_seed777",
    exp_df=exp_df_mul,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/60 [00:00<?, ?case/s]

In [22]:
results_mul

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,KGA,6,2,6,0.474497,0.115660,0.833333,0.350000,0.625000,...,0.632479,0.388889,0.166667,0.333333,0.000000,0.076923,0.000000,0.466870,0.000000,0.500000
1,0,FC,7,2,6,0.399450,0.148900,0.650000,0.380769,0.617500,...,0.693590,0.450000,0.166667,0.333333,0.000000,0.076923,0.000000,0.465868,0.000000,0.500000
2,0,GDA,8,2,7,0.473716,0.197432,0.750000,0.396154,0.650000,...,0.966561,0.500000,0.166667,0.333333,0.000000,0.076923,0.222971,0.682569,0.000000,0.500000
3,0,RHA,9,2,8,0.458771,0.117543,0.800000,0.407692,0.737500,...,0.862405,0.541667,0.166893,0.333333,0.000452,0.153846,0.000000,0.470700,0.000000,0.500000
4,0,YW,10,2,9,0.429829,0.126326,0.733333,0.330769,0.761538,...,0.884383,0.576923,0.166667,0.333333,0.000000,0.076923,0.063870,0.530888,0.000000,0.500000
5,0,GMA,13,2,12,0.441078,0.098824,0.783333,0.411538,0.779687,...,0.982505,0.656250,0.172409,0.333333,0.011485,0.153846,0.000000,0.467780,0.000000,0.500000
6,0,TIA,15,2,14,0.501639,0.186611,0.816667,0.476923,0.831944,...,1.076923,0.833333,0.166667,0.333333,0.000000,0.076923,0.000000,0.847657,0.000000,1.000000
7,0,GS,16,2,15,0.386285,0.205903,0.566667,0.380769,0.772368,...,1.073311,0.710526,0.040318,0.000000,0.080637,0.076923,0.245543,0.707166,0.000000,0.500000
8,0,GG,17,2,16,0.420572,0.091144,0.750000,0.361538,0.845000,...,1.046196,0.725000,0.167350,0.333333,0.001366,0.153846,0.000000,0.466969,0.000000,0.500000
9,0,XFA,18,2,16,0.428554,0.073775,0.783333,0.311538,0.808333,...,0.934066,0.690476,0.166667,0.333333,0.000000,0.076923,0.000000,0.471234,0.000000,0.500000


### --- Cleanup ---

In [23]:
sys.stdout = original_stdout
log_file.close()